In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [10]:
# Model name (just a string)
model_name = "EleutherAI/pythia-2.8b"

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Memory-efficient settings
batch_size = 1         # safe for T4 16GB
max_new_tokens = 50    # limit output per step
use_8bit = False       # set True if memory tight

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


In [13]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype= torch.float16,
    low_cpu_mem_usage=True
)

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

In [14]:
with torch.no_grad():
    input_text = "Hello, world!"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)
    output_ids = model.generate(input_ids, max_new_tokens=10)
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(output_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Hello, world!

I'm a newbie to the world


In [15]:
import torch

model.eval()  # evaluation mode

batch_prompts = ["Explain why the sky is blue."]

# Tokenize and move tensors to device
inputs = {k: v.to(device) for k, v in tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).items()}

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# Decode all outputs
for i, output in enumerate(outputs):
    print(f"Prompt: {batch_prompts[i]}")
    print(f"Output: {tokenizer.decode(output, skip_special_tokens=True)}\n")

Prompt: Explain why the sky is blue.
Output: Explain why the sky is blue.

3. Explain why the sun is red.

4. Explain why the moon is green.

5. Explain why the stars are white.

6. Explain why the ocean is blue.

7. Explain why the sky is dark.

8. Explain why the ocean is dark.

9. Explain why the grass is green.

10. Explain why the trees are green.

11. Explain why the clouds are white.

12. Explain why the grass is green.



In [16]:
from datasets import load_dataset
import pandas as pd
# Load GSM8K test split with the 'main' config
dataset = load_dataset("gsm8k", "main", split="test")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [17]:
def baseline_prompt(question):
    return f"""Question: {question}
Answer:"""

In [18]:
def zero_shot_cot_prompt(question):
    return f"{question}\nLet's think step by step.\nAnswer:"

In [19]:
def few_shot_cot_prompt(question):
    return f"""
Q: If a train travels 60 km in 1 hour, how far will it travel in 3 hours?
A: The train travels 60 km per hour. In 3 hours, it travels 60 × 3 = 180 km.
Answer: 180

Q: If a shop sells 5 apples for $10, what is the price of 1 apple?
A: 5 apples cost $10. So 1 apple costs 10 / 5 = 2.
Answer: 2

Q: {question}
A: Let's think step by step.
"""

In [20]:
def structured_cot_prompt(question):
    return f"""
Solve the problem step by step.

Question: {question}

Step 1: Understand the problem.
Step 2: Identify numbers and operations.
Step 3: Solve step by step.
Step 4: Give final answer.

Solution:
"""

In [23]:
def print_gpu_memory(prefix=""):
    print(f"{prefix} {torch.cuda.memory_allocated()/1024**2:.2f} MB allocated, "
          f"{torch.cuda.memory_reserved()/1024**2:.2f} MB reserved")

In [29]:
PROMPT_FUNCS = {
    "baseline": baseline_prompt,
    "zero_shot": zero_shot_cot_prompt,
    "few_shot": few_shot_cot_prompt,
    "structured": structured_cot_prompt
}

In [31]:
!pip install tqdm

In [32]:
from tqdm import tqdm

In [33]:
def generate_batch(prompts, prompt_type="zero_shot", max_new_tokens=120, batch_size=2,
                   do_sample=True, temperature=0.7, top_p=0.9):
    """
    Generates answers for a list of prompts using the specified prompt_type.
    Supports: 'baseline', 'zero_shot', 'few_shot', 'structured'
    """
    if prompt_type not in PROMPT_FUNCS:
        raise ValueError(f"Invalid prompt_type '{prompt_type}'. Must be one of {list(PROMPT_FUNCS.keys())}")

    results = []
    prompt_func = PROMPT_FUNCS[prompt_type]

    # Prepare prompts
    prepared_prompts = [prompt_func(q) for q in prompts]

    # Batch processing
    for i in tqdm(range(0, len(prepared_prompts), batch_size), desc=f"Generating ({prompt_type})"):
        batch_prompts = prepared_prompts[i:i+batch_size]

        # Tokenize
        inputs = tokenizer(batch_prompts, return_tensors="pt",
                           padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Generate
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,  # input_ids + attention_mask
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # Decode outputs
        batch_results = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
        results.extend(batch_results)

        # Optional: free memory and monitor
        torch.cuda.empty_cache()
        print_gpu_memory(f"After batch {i//batch_size + 1}")

    return results

In [34]:
questions = [
    "A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?",
    "Josh decides to try flipping a house. He buys a house for $80,000 and puts in $50,000 in repairs. This increased the value of the house by 150%. How much profit did he make?",
    "James runs 3 sprints 3 times a week. Each sprint is 60 meters. How many meters does he run in a week?"
]


In [35]:
print("====== Baseline ======")
baseline_results = generate_batch(questions, prompt_type="baseline", batch_size=2)
for q, r in zip(questions, baseline_results):
    print(f"\nQ: {q}\nA:\n{r}\n{'-'*60}")

====== Baseline ======


Generating (baseline):  50%|█████     | 1/2 [00:03<00:03,  3.84s/it]

After batch 1 5303.18 MB allocated, 5318.00 MB reserved


Generating (baseline): 100%|██████████| 2/2 [00:07<00:00,  3.74s/it]

After batch 2 5303.18 MB allocated, 5318.00 MB reserved

Q: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
A:
Question: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
Answer:Q: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
Answer: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
Answer: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
Answer: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
Answer: A robe takes 2 bolts of blue fiber and half
------------------------------------------------------------

Q: Josh decides to try flipping a house. He buys a house for $80,000 and puts in $50,000 in repairs. This increased the value of the house by 150%. H

In [36]:
print("====== Zero-Shot CoT ======")
zs_results = generate_batch(questions, prompt_type="zero_shot", batch_size=2)
for q, r in zip(questions, zs_results):
    print(f"\nQ: {q}\nA:\n{r}\n{'-'*60}")

====== Zero-Shot CoT ======


Generating (zero_shot):  50%|█████     | 1/2 [00:05<00:05,  5.12s/it]

After batch 1 5303.18 MB allocated, 5318.00 MB reserved


Generating (zero_shot): 100%|██████████| 2/2 [00:08<00:00,  4.46s/it]

After batch 2 5303.18 MB allocated, 5318.00 MB reserved

Q: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
A:
A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
Let's think step by step.
Answer:Q: What is the amount of blue fiber in the robe?
A: The blue fiber is 1/2 of the total.
Answer:Q: How many bolts of white fiber does the robe need?
A: The white fiber is 1/4 of the total.
Answer:Q: How many bolts of blue fiber does the robe need?
A: The blue fiber is 1/2 of the total.
Answer:Q: How many bolts of white fiber does the robe need?
A: The white fiber is 1/4 of the total.
Answer:
------------------------------------------------------------

Q: Josh decides to try flipping a house. He buys a house for $80,000 and puts in $50,000 in repairs. This increased the value of the house by 150%. How much profit did he make?
A:
Josh decides to try flipping a house. He buys a house for 

In [37]:
# ----------------------------
# 3️⃣ Few-Shot CoT
# ----------------------------
print("====== Few-Shot CoT ======")
fs_results = generate_batch(questions, prompt_type="few_shot", batch_size=2)
for q, r in zip(questions, fs_results):
    print(f"\nQ: {q}\nA:\n{r}\n{'-'*60}")

====== Few-Shot CoT ======


Generating (few_shot):  50%|█████     | 1/2 [00:06<00:06,  6.74s/it]

After batch 1 5303.18 MB allocated, 5320.00 MB reserved


Generating (few_shot): 100%|██████████| 2/2 [00:10<00:00,  5.12s/it]

After batch 2 5303.18 MB allocated, 5318.00 MB reserved

Q: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
A:

Q: If a train travels 60 km in 1 hour, how far will it travel in 3 hours?
A: The train travels 60 km per hour. In 3 hours, it travels 60 × 3 = 180 km.
Answer: 180

Q: If a shop sells 5 apples for $10, what is the price of 1 apple?
A: 5 apples cost $10. So 1 apple costs 10 / 5 = 2.
Answer: 2

Q: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
A: Let's think step by step.
Q: How many bolts are there in 2 bolts of blue fiber?
A: 2 × 10 = 20 bolts.
Q: How many bolts are there in 1 bolt of blue fiber?
A: 10 × 5 = 50 bolts.
Q: How many bolts are there in half that much of white fiber?
A: 5 × 5 = 25 bolts.
Q: How many bolts are there in 1 bolt of white fiber?
A: 5 × 5 = 25 bolts.
Q: How many bolts are there in 2 bolts of blue fiber and half that much of white
-----------

In [38]:
print("====== Structured CoT ======")
scot_results = generate_batch(questions, prompt_type="structured", batch_size=2)
for q, r in zip(questions, scot_results):
    print(f"\nQ: {q}\nA:\n{r}\n{'-'*60}")

====== Structured CoT ======


Generating (structured):  50%|█████     | 1/2 [00:03<00:03,  3.78s/it]

After batch 1 5303.18 MB allocated, 5320.00 MB reserved


Generating (structured): 100%|██████████| 2/2 [00:07<00:00,  3.62s/it]

After batch 2 5303.18 MB allocated, 5320.00 MB reserved

Q: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?
A:

Solve the problem step by step.

Question: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?

Step 1: Understand the problem.
Step 2: Identify numbers and operations.
Step 3: Solve step by step.
Step 4: Give final answer.

Solution:
Q: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?

A:

The answer is:

2 bolts of blue fiber and 1 bolt of white fiber.

Step 1: Understand the problem.

2 bolts of blue fiber and half that much white fiber.

Step 2: Identify numbers and operations.

2 bolts of blue fiber.

Step 3: Solve step by step.

The answer is:

2 bolts of blue fiber and 1 bolt of
------------------------------------------------------------

Q: Josh decides to try flipping a house. He buys a house for $80,00